# Geolocation table: cleaning

Geolocation table cleaning script
Source : raw/olist_geolocation_dataset.csv   (1,000,163 rows, many points per zip)
Output : cleaned/geolocation_clean.csv       (19,015 rows: ONE row per zip prefix, 6 columns)
Needs  : cleaned/customers_clean.csv and cleaned/sellers_clean.csv for the coverage check.
Run cell by cell in VS Code (# %% markers), or as a whole script from the project folder.

Decisions and findings for this table are written up in `cleaning_notes/README_geolocation.md`.

### Setup

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
RAW = Path('raw')
CLEAN = Path('cleaned')
CLEAN.mkdir(exist_ok=True)
zkey = 'geolocation_zip_code_prefix'

### 1. Load (zip prefix as text so leading zeros survive)

In [2]:
geo = pd.read_csv(RAW / 'olist_geolocation_dataset.csv', dtype={zkey: str})
geo[zkey] = geo[zkey].str.zfill(5)
print(geo.shape)                          # (1000163, 5)
print(geo.isna().sum())                   # no nulls
print(geo.duplicated().sum())             # 261,831 exact repeats

(1000163, 5)
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64
261831


### 2. Drop exact repeated rows (same point repeated: no information lost)

In [3]:
geo = geo.drop_duplicates().reset_index(drop=True)
print(geo.shape)                          # (738332, 5)
print(geo[zkey].nunique())                # 19,015 distinct zips

(738332, 5)
19015


### 3. Flag points outside Brazil (27 rows): bad geocodes, not repaired

In [4]:
in_brazil = geo['geolocation_lat'].between(-34, 5.5) & geo['geolocation_lng'].between(-74, -28)
geo['coord_outside_brazil'] = ~in_brazil
print(geo['coord_outside_brazil'].sum())  # 27

27


### 4. City text: odd apostrophe, accents, garbled characters

In [5]:
c = geo['geolocation_city'].str.replace('´', "'", regex=False)
c = c.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('ascii').str.strip()
geo['geolocation_city'] = c
print(geo['geolocation_city'].nunique())  # 5,967 (was 8,011 before dedup and accent removal)

5967


### 5. How consistent is each zip?

In [6]:
g = geo.groupby(zkey)
print((g['geolocation_state'].nunique() > 1).sum())   # 8 zips with more than one state
print((g['geolocation_city'].nunique() > 1).sum())    # 549 zips with more than one city name
print(g.size().describe())                            # points per zip

8
549
count    19015.000000
mean        38.828925
std         50.875735
min          1.000000
25%          8.000000
50%         23.000000
75%         49.000000
max        779.000000
dtype: float64


### 6. Build ONE row per zip

In [7]:
mode = lambda s: s.mode().iloc[0]
state = geo.groupby(zkey)['geolocation_state'].agg(mode)      # most common state
city = geo.groupby(zkey)['geolocation_city'].agg(mode)        # most common city name

# coordinates: median of valid points that sit in the zip's winning state
pts = geo.loc[~geo['coord_outside_brazil']].copy()
pts = pts[pts['geolocation_state'] == pts[zkey].map(state)]
coords = pts.groupby(zkey)[['geolocation_lat', 'geolocation_lng']].median()

points_count = geo.groupby(zkey).size().rename('points_count')
geo_zip = pd.concat([city, state, coords, points_count], axis=1).reset_index()

print(geo_zip.shape)                       # (19015, 6)
print(geo_zip[zkey].is_unique)             # True
print(geo_zip.isna().sum())                # 4 zips without coordinates
print(geo_zip['geolocation_city'].nunique())   # 5,771
print(geo_zip[geo_zip['geolocation_lat'].isna()].to_string())

(19015, 6)
True
geolocation_zip_code_prefix    0
geolocation_city               0
geolocation_state              0
geolocation_lat                4
geolocation_lng                4
points_count                   0
dtype: int64
5771
      geolocation_zip_code_prefix         geolocation_city geolocation_state  geolocation_lat  geolocation_lng  points_count
6077                        18243  bom retiro da esperanca                SP              NaN              NaN             1
15781                       78131            varzea grande                MT              NaN              NaN             1
16419                       83252       ilha dos valadares                PR              NaN              NaN             1
18309                       95130      santa lucia do piai                RS              NaN              NaN             1


### 7. Coverage of customers and sellers

In [8]:
cust = pd.read_csv(CLEAN / 'customers_clean.csv', dtype={'customer_zip_code_prefix': str},
                   usecols=['customer_zip_code_prefix'])
sell = pd.read_csv(CLEAN / 'sellers_clean.csv', dtype={'seller_zip_code_prefix': str},
                   usecols=['seller_zip_code_prefix'])
has = geo_zip.loc[geo_zip['geolocation_lat'].notna(), zkey]
print((~cust['customer_zip_code_prefix'].isin(has)).sum())                     # 279 customer rows
print(cust.loc[~cust['customer_zip_code_prefix'].isin(has), 'customer_zip_code_prefix'].nunique())  # 158 zips
print((~sell['seller_zip_code_prefix'].isin(has)).sum())                       # 7 seller rows

279
158
7


### 8. Save and verify (keep this cell LAST)

In [9]:
geo_zip.to_csv(CLEAN / 'geolocation_clean.csv', index=False)

check = pd.read_csv(CLEAN / 'geolocation_clean.csv', dtype={zkey: str})
print(check.shape)                                   # (19015, 6)
print(check[zkey].is_unique)                         # True
print(check[zkey].str.len().value_counts())          # all 5
print(check['geolocation_lat'].isna().sum())         # 4
print(check[['geolocation_lat', 'geolocation_lng']].describe())

(19015, 6)
True
geolocation_zip_code_prefix
5    19015
Name: count, dtype: int64
4
       geolocation_lat  geolocation_lng
count     19011.000000     19011.000000
mean        -19.078966       -46.077180
std           7.240744         5.198294
min         -33.690729       -72.909444
25%         -23.564539       -49.008455
50%         -22.425826       -46.632757
75%         -15.617623       -43.253402
max           4.478429       -32.412876
